### Previsão de Demanda com Random Forest 


Este notebook utiliza a base de features criada no notebook anterior para treinar um modelo de Machine Learning.

O objetivo é prever o total de vendas esperado até o fim do dia, usando informações parciais do próprio dia, como vendas por hora, receita acumulada e tempo restante.

Modelo escolhido: Random Forest Regressor.

In [0]:
import logging

In [0]:
%run /Workspace/Users/kalitamariano01@gmail.com/merca-data-platform/notebooks/utils/utils_feat_squad2_99_helpers

## Leitura Da Base De Features

A tabela usada neste notebook foi criada no notebook de Feature Engineering.

Ela contém os dados já tratados, padronizados e transformados em variáveis numéricas para o modelo.

In [0]:
df_features_spark = spark.table("squad2.ml_features_previsao_demanda_vendas_hora_v1")

display(df_features_spark.limit(10))

## Preparação Dos Dados Para O Modelo

Nesta etapa, a base Spark é convertida para Pandas para uso com o Scikit-Learn.

As features representam o comportamento das vendas ao longo do dia.

A variável alvo é `total_vendas_dia`, que representa o total real de vendas do dia inteiro.

In [0]:

df_modelo = df_features_spark.toPandas()

df_modelo = df_modelo.dropna()

features = [
    "hora_evento",
    "dia_semana",
    "vendas_hora",
    "receita_hora",
    "ticket_medio_hora",
    "vendas_acumuladas",
    "receita_acumulada",
    "tempo_restante_dia",
    "percentual_dia_decorrido",
]

target = "total_vendas_dia"

X = df_modelo[features]
y = df_modelo[target]

print("Total de linhas:", len(df_modelo))
print("Features usadas:", features)
print("Target:", target)

## Divisão Entre Treino E Teste

A base foi dividida em treino e teste.

- 80% dos dados são usados para treinar o modelo.
- 20% dos dados são usados para avaliar o desempenho.

O `random_state=42` garante que o resultado seja reproduzível.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## Treinamento Do Random Forest Regressor

O Random Forest Regressor cria várias árvores de decisão e combina os resultados para gerar uma previsão mais estável.

Ele foi escolhido porque consegue capturar relações não lineares entre as variáveis, como horário, vendas acumuladas, receita acumulada e comportamento diário de demanda.

In [0]:
modelo_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

print("Modelo Random Forest treinado com sucesso.")

## Avaliação Do Modelo

Foram usadas quatro métricas para avaliar o desempenho do Random Forest:

- **MAE**: erro médio absoluto. Mostra, em média, quantas vendas o modelo erra na previsão.
- **RMSE**: penaliza erros maiores. Quando fica muito acima do MAE, indica que existem alguns casos com erro mais alto.
- **MAPE**: erro percentual médio. Mostra o erro do modelo em porcentagem, facilitando a comparação com outros modelos.
- **R²**: mostra quanto o modelo explica da variação dos dados. Quanto mais próximo de 1, melhor.

Essas métricas foram calculadas no conjunto de teste, ou seja, em dados que o modelo não usou durante o treinamento.

In [0]:
predicoes = modelo_rf.predict(X_test)

mae = mean_absolute_error(y_test, predicoes)
rmse = np.sqrt(mean_squared_error(y_test, predicoes))
r2 = r2_score(y_test, predicoes)
mape = np.mean(np.abs((y_test - predicoes) / y_test)) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"R²: {r2:.2f}")

In [0]:
predicoes_treino = modelo_rf.predict(X_train)
predicoes_teste = modelo_rf.predict(X_test)

mae_treino = mean_absolute_error(y_train, predicoes_treino)
rmse_treino = np.sqrt(mean_squared_error(y_train, predicoes_treino))
mape_treino = np.mean(np.abs((y_train - predicoes_treino) / y_train)) * 100
r2_treino = r2_score(y_train, predicoes_treino)

mae_teste = mean_absolute_error(y_test, predicoes_teste)
rmse_teste = np.sqrt(mean_squared_error(y_test, predicoes_teste))
mape_teste = np.mean(np.abs((y_test - predicoes_teste) / y_test)) * 100
r2_teste = r2_score(y_test, predicoes_teste)

df_metricas = pd.DataFrame({
    "Base": ["Treino", "Teste"],
    "MAE": [mae_treino, mae_teste],
    "RMSE": [rmse_treino, rmse_teste],
    "MAPE (%)": [mape_treino, mape_teste],
    "R²": [r2_treino, r2_teste]
})

display(df_metricas)

## Interpretação Dos Resultados

O modelo apresentou **MAPE de 10,06%** e **R² de 0,98** no conjunto de teste.

Isso indica que o Random Forest conseguiu capturar bem o padrão geral da demanda. Em média, o erro percentual ficou próximo de 10%, enquanto o R² mostrou que o modelo explica cerca de 98% da variação dos dados.

O RMSE ficou maior que o MAE, indicando que existem alguns períodos em que o erro é mais alto. Isso pode acontecer em dias com comportamento atípico ou com poucas janelas de venda.


## Resultado Das Previsões

Nesta etapa, juntamos os valores reais e previstos para analisar o desempenho do modelo linha a linha.

In [0]:
df_resultado = X_test.copy()
df_resultado["total_vendas_real"] = y_test.values
df_resultado["total_vendas_previsto"] = predicoes
df_resultado["erro_absoluto"] = abs(
    df_resultado["total_vendas_real"] - df_resultado["total_vendas_previsto"]
)

display(df_resultado.head(20))

## Importância Das Features

A importância das features mostra quais variáveis mais influenciaram o modelo na previsão do total de vendas do dia.

In [0]:
df_importancias = pd.DataFrame({
    "feature": features,
    "importancia": modelo_rf.feature_importances_
}).sort_values("importancia", ascending=False)

display(df_importancias)

## Salvamento Dos Resultados

Os resultados do modelo são salvos em tabelas para serem usados no notebook de insights e posteriormente no dashboard.

In [0]:
df_resultado_spark = spark.createDataFrame(df_resultado)
df_importancias_spark = spark.createDataFrame(df_importancias)

(
    df_resultado_spark
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("squad2.ml_resultado_previsao_demanda_rf_v1")
)

(
    df_importancias_spark
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("squad2.ml_importancia_features_previsao_demanda_rf_v1")
)

print("Tabelas salvas:")
print("squad2.ml_resultado_previsao_demanda_rf_v1")
print("squad2.ml_importancia_features_previsao_demanda_rf_v1")

In [0]:
importancias = pd.DataFrame({
    "feature": features,
    "importancia": modelo_rf.feature_importances_
}).sort_values("importancia", ascending=False)

#display(importancias)